<a href="https://colab.research.google.com/github/reynaudnangue28/test/blob/main/CO2_Emission_Project_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Regression Models
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Google Colab to access and mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ---  load a mapping csv---
try:
    mapping_df = pd.read_csv('/content/drive/MyDrive/Data/EEA2024CO2Data/eea_column_mapping_with_nulls.csv')
    cols_to_keep = mapping_df[mapping_df['keep'] == True]
    rename_map = dict(zip(cols_to_keep['column_name_original'], cols_to_keep['column_name_new']))
except FileNotFoundError:
    print("Error : file is missing.")
    exit()

# ---  load the EEA file  ---

df1 = pd.read_csv(
    "/content/drive/MyDrive/Data/EEA2024CO2Data/data.csv",
    nrows=7000000,
    low_memory=False
)

#Reading Between Raws 7 million and 10.8 million

df2 = pd.read_csv(
    "/content/drive/MyDrive/Data/EEA2024CO2Data/data.csv",
    skiprows=range(1, 7_000_001),  # skip header + first 7M rows
    low_memory=False
)

# ---  Filter and rename ---
# choose only the columns with 'keep=True'
df1 = df1[rename_map.keys()]
# Rename the columns
df1 = df1.rename(columns=rename_map)

# choose only the columns with 'keep=True'
df2 = df2[rename_map.keys()]
# Rename the columns
df2 = df2.rename(columns=rename_map)

#Concat of df1 and df2
df = pd.concat([df1, df2], axis=0, ignore_index=True)


fuel_map   = {
    'petrol': "Petrol",
    'diesel': "Diesel",
    'petrol/electric': "Petrol-Electric Hybrid",
    'diesel/electric': "Diesel-Electric Hybrid",
    'electric': "100% Electric",
    'lpg': "LPG",
    'ng': "Natural Gas",
    'e85': "Ethanol 85%",
    'hydrogen': "Hydrogen",
    'unknown': "Unknown",
    'es': "Petrol"
}

df['fuel_type'] = df['fuel_type'].str.strip().str.lower().replace(fuel_map)

#  Basic Preprocessing

#numeric features
features = [
    "mass_kg",
    "wltp_mass_kg",
    "engine_cc",
    "power_kw",
    "electric_consumption",
    "fuel_type"
]
target = 'co2_wltp'

df['engine_cc'] = df['engine_cc'].fillna(0)


df['electric_consumption'] = df['electric_consumption'].fillna(0)

# Drop rows with missing values in these specific columns
df = df.dropna(subset=features + [target])
X = pd.get_dummies(df[features], columns=['fuel_type'], drop_first=True)
y = df[target]


#  Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#  Initialize and Train Models

lr_model = LinearRegression()
lrr_model= Ridge(alpha=1.0)
lrl_model = Lasso(alpha=0.1)
lre_model = ElasticNet(alpha=0.1, l1_ratio=0.5)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
et_model = ExtraTreesRegressor(n_estimators=100, random_state=42)
dt_model = DecisionTreeRegressor(max_depth=10, random_state=42)
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

knn_model = KNeighborsRegressor(n_neighbors=5)
svr_model = SVR(kernel='rbf', C=100, epsilon=0.1)

nn_model = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(hidden_layer_sizes=(100, 50, 25),
                         activation='relu',
                         solver='adam',
                         alpha=0.001, # Regularization to prevent overfitting
                         learning_rate='adaptive',
                         max_iter=500,
                         random_state=42))
])

lr_model.fit(X_train, y_train)
lrr_model.fit(X_train, y_train)
lrl_model.fit(X_train, y_train)
lre_model.fit(X_train, y_train)

rf_model.fit(X_train, y_train)
et_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

knn_model.fit(X_train, y_train)
svr_model.fit(X_train, y_train)

nn_model.fit(X_train, y_train)

#  Evaluation Function
def evaluate_model(model, X_test, y_test, name):
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    print(f"--- {name} Performance ---")
    print(f"MAE (Mean Absolute Error): {mae:.2f} g/km")
    print(f"RMSE (Root Mean Squared Error): {rmse:.2f} g/km")
    print(f"R² score(Coefficient of Determination): {r2:.4f}")

#  Compare Results

print("1. Linear Models (Fast & Interpretable)")
evaluate_model(lr_model, X_test, y_test, "Linear Regression")
evaluate_model(lrr_model, X_test, y_test, "Ridge Regression")
evaluate_model(lrl_model, X_test, y_test, "Lasso Regression")
evaluate_model(lre_model, X_test, y_test, "ElasticNet")

print("2. Tree-Based & Ensemble Models (High Accuracy)")
evaluate_model(rf_model, X_test, y_test, "Random Forest")
evaluate_model(et_model, X_test, y_test, "Extra Trees")
evaluate_model(dt_model, X_test, y_test, "Decision Tree")
evaluate_model(gb_model, X_test, y_test, "Gradient Boosting")
evaluate_model(xgb_model, X_test, y_test, "XGBoost")
"""
I skip these models because it takes a lot of time to run
print("3. Instance-Based & Support Vector Models")
evaluate_model(knn_model, X_test, y_test, "K-Nearest Neighbors (KNN) Regression")
evaluate_model(svr_model, X_test, y_test, "Support Vector Regression (SVR)")
"""
print("4. Neural Networks")
evaluate_model(nn_model, X_test, y_test, "Neural Network")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Neuer Abschnitt

In [ ]:
!pip install opencv-python
!pip install scikit-optimize


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 2.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
